In [2]:
import pickle

In [3]:
with open("data", "rb") as f:
    data = pickle.load(f) 

In [4]:
data

{'good':         MS2.mz  MS2.intensity IO  CE                                 name
 0    53.294861    2741.973633  +  20  1-(3,4-dichlorophenyl)-3-methylurea
 1    55.329395    2350.334961  +  20  1-(3,4-dichlorophenyl)-3-methylurea
 2    57.623466    2424.935059  +  20  1-(3,4-dichlorophenyl)-3-methylurea
 3    58.028690  212608.359375  +  20  1-(3,4-dichlorophenyl)-3-methylurea
 4    60.391148    3166.069580  +  20  1-(3,4-dichlorophenyl)-3-methylurea
 ..         ...            ... ..  ..                                  ...
 90  307.174622    3951.933594  +  35                            valsartan
 91  345.195312    4217.800781  +  35                            valsartan
 92  352.176056    5398.227539  +  35                            valsartan
 93  362.222046   11204.687500  +  35                            valsartan
 94  424.453461    2664.944580  +  35                            valsartan
 
 [12411 rows x 5 columns],
 'bad':         MS2.mz  MS2.intensity IO  CE                   

In [5]:
type(data)

dict

In [7]:
len(data)

2

In [14]:
len(data['good']), type(data['good'])

(12411, pandas.core.frame.DataFrame)

In [9]:
len(data['bad'])

38066

In [10]:
data['good']

,MS2.mz,MS2.intensity,IO,CE,name
0,53.294861,2741.973633,+,20,"1-(3,4-dichlorophenyl)-3-methylurea"
1,55.329395,2350.334961,+,20,"1-(3,4-dichlorophenyl)-3-methylurea"
2,57.623466,2424.935059,+,20,"1-(3,4-dichlorophenyl)-3-methylurea"
3,58.028690,212608.359375,+,20,"1-(3,4-dichlorophenyl)-3-methylurea"
4,60.391148,3166.069580,+,20,"1-(3,4-dichlorophenyl)-3-methylurea"
...,...,...,...,...,...
90,307.174622,3951.933594,+,35,valsartan
91,345.195312,4217.800781,+,35,valsartan
92,352.176056,5398.227539,+,35,valsartan
93,362.222046,11204.687500,+,35,valsartan


"data" is a dictionary that keeps "good" and "bad" spectra. The indices are from 0 to the number of peaks in each spectra.

In [16]:
data['good'].index

Int64Index([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9,
            ...
            85, 86, 87, 88, 89, 90, 91, 92, 93, 94],
           dtype='int64', length=12411)

In [17]:
data['good'].index[100]

7

In [35]:
bad = data.get("bad").copy()
good = data.get("good").copy()

In [39]:
bad

,MS2.mz,MS2.intensity,IO,CE,name
0,56.843941,4600.507324,+,10,"1-(3,4-dichlorophenyl)-3-methylurea"
1,58.028500,60621.242188,+,10,"1-(3,4-dichlorophenyl)-3-methylurea"
2,73.529976,4858.581055,+,10,"1-(3,4-dichlorophenyl)-3-methylurea"
3,75.155380,4815.912598,+,10,"1-(3,4-dichlorophenyl)-3-methylurea"
4,75.415131,5568.496582,+,10,"1-(3,4-dichlorophenyl)-3-methylurea"
...,...,...,...,...,...
81,258.161346,7698.964355,+,50,Venlafaxine
82,322.403229,8803.409180,+,50,Venlafaxine
83,323.133118,7428.805664,+,50,Venlafaxine
84,337.984741,7633.589844,+,50,Venlafaxine


In [37]:
import numpy as np

create lists with arrays that represent the indices of every instance

In [114]:
index_arrays_b = np.split(np.arange(len(bad)), np.where(bad.index==0)[0][1:])
index_arrays_g = np.split(np.arange(len(good)), np.where(good.index==0)[0][1:])

In [115]:
len(index_arrays_b), len(index_arrays_g)

(578, 261)

In [121]:
index_arrays_g[0], index_arrays_g[1]

(array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16,
        17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]),
 array([30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46,
        47, 48, 49, 50, 51, 52, 53, 54]))

In [88]:
import torch

create the data matrix

In [145]:
def create_data_matrix(list_b, list_g):
    M = []
    
    for arr in list_b:
        df = bad.iloc[arr][["MS2.mz","MS2.intensity"]].copy()
        observation = {"X": df.to_numpy(), "Y": np.zeros((1,1),dtype=np.int64)}
        observation["X"] = torch.from_numpy(observation["X"]).unsqueeze(0)
        observation["Y"] = torch.from_numpy(observation["Y"])
        M.append(observation)
    
    for arr in list_g:
        df = good.iloc[arr][["MS2.mz", "MS2.intensity"]].copy()
        observation = {"X": df.to_numpy(), "Y": np.ones((1,1), dtype=np.int64)}
        observation["X"] = torch.from_numpy(observation["X"]).unsqueeze(0)
        observation["Y"] = torch.from_numpy(observation["Y"])
        M.append(observation)
    
    return M    

In [146]:
M = create_data_matrix(index_arrays_b, index_arrays_g)

In [147]:
len(M)

839

In [148]:
M[0]

{'X': tensor([[[5.6844e+01, 4.6005e+03],
          [5.8028e+01, 6.0621e+04],
          [7.3530e+01, 4.8586e+03],
          [7.5155e+01, 4.8159e+03],
          [7.5415e+01, 5.5685e+03],
          [9.1673e+01, 6.1677e+03],
          [9.7675e+01, 5.2213e+03],
          [1.1640e+02, 4.7816e+03],
          [1.2702e+02, 1.0066e+05],
          [1.3913e+02, 4.5533e+03],
          [1.6098e+02, 8.7798e+03],
          [1.6199e+02, 2.3103e+06],
          [1.6299e+02, 8.8226e+03],
          [1.7511e+02, 5.4293e+03],
          [1.8088e+02, 5.3959e+03],
          [1.8797e+02, 1.1656e+04],
          [2.1809e+02, 8.8399e+03],
          [2.1901e+02, 1.5031e+07],
          [2.1917e+02, 1.0843e+04],
          [2.1994e+02, 9.4077e+03],
          [2.2001e+02, 3.8139e+04],
          [2.3702e+02, 6.0126e+03],
          [3.4830e+02, 5.5527e+03]]], dtype=torch.float64),
 'Y': tensor([[0]])}

In [ ]:
def extract_features(Mx):
    np.random.shuffle(Mx)
    data_X = []